# 11 · The solver toolbox

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=11-solver-toolbox.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/11-solver-toolbox.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>


Assembling gives a linear system $A\,x = b$. So far we wrote
`a.mat.Inverse(fes.FreeDofs())` — a **direct** solver, exact but memory-hungry
for large 3D problems. Here we open the toolbox: free dofs, **iterative**
solvers with **preconditioners**, and **static condensation**. We use a classic
L-shaped domain.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import WorkPlane, OCCGeometry
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.krylovspace import CGSolver

def Lshape(maxh):
    shp = (WorkPlane().MoveTo(0, 0).LineTo(1, 0).LineTo(1, 0.5)
           .LineTo(0.5, 0.5).LineTo(0.5, 1).LineTo(0, 1).Close().Face())
    return Mesh(OCCGeometry(shp, dim=2).GenerateMesh(maxh=maxh))

mesh = Lshape(0.05)

## 1. Free dofs

`H1(..., dirichlet=...)` marks the constrained boundary dofs. `fes.FreeDofs()`
is the bit-array of the *remaining* (free) unknowns — the ones we actually
solve for. The solver must be told about them.

In [ ]:
fes = H1(mesh, order=3, dirichlet=".*")
u, v = fes.TnT()
print(f"total dofs = {fes.ndof}, free dofs = {sum(fes.FreeDofs())}")

a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(1*v*dx).Assemble()
gfu = GridFunction(fes)

## 2. Direct solver

`Inverse(freedofs, inverse="sparsecholesky")` factorises the (sparse, SPD)
matrix. Exact, robust, great for small/medium problems.

In [ ]:
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
Draw(gfu, mesh)

## 3. Iterative solver + preconditioner

Conjugate gradients (`CGSolver`) needs only matrix–vector products plus a
**preconditioner** that approximates $A^{-1}$. The number of iterations is the
real quality measure. (We point the coarse solver of `multigrid`/`bddc` at
`sparsecholesky`.)

In [ ]:
def solve_cg(mesh, ptype, **kw):
    fes = H1(mesh, order=3, dirichlet=".*")
    u, v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx)
    pre = Preconditioner(a, ptype, **kw)          # registered BEFORE assembly!
    a.Assemble()
    f = LinearForm(1*v*dx).Assemble()
    gfu = GridFunction(fes)
    inv = CGSolver(a.mat, pre.mat, maxiter=5000, tol=1e-10)
    gfu.vec.data = inv * f.vec
    return fes.ndof, inv.iterations

for ptype, kw in [("local", {}), ("bddc", {"inverse": "sparsecholesky"}),
                  ("multigrid", {"inverse": "sparsecholesky"})]:
    ndof, its = solve_cg(mesh, ptype, **kw)
    print(f"CG + {ptype:9s}: {its:3d} iterations")

### Watching the residual fall

A `callback` on `CGSolver` is handed the iteration number and the **residual
norm** at every step. Collecting those and plotting them on a log scale shows
not just *how many* steps each preconditioner needs, but *how* it drives the
residual down.

In [ ]:
import matplotlib.pyplot as plt

def cg_residuals(ptype, **kw):
    fes = H1(mesh, order=3, dirichlet=".*")
    u, v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx)
    pre = Preconditioner(a, ptype, **kw)
    a.Assemble()
    f = LinearForm(1*v*dx).Assemble()
    gfu = GridFunction(fes)
    res = []
    inv = CGSolver(a.mat, pre.mat, maxiter=5000, tol=1e-10,
                   callback=lambda k, r: res.append(r))     # record |residual| per step
    gfu.vec.data = inv * f.vec
    return res

plt.figure(figsize=(6, 3.3))
for ptype, kw in [("local", {}), ("bddc", {"inverse": "sparsecholesky"}),
                  ("multigrid", {"inverse": "sparsecholesky"})]:
    r = cg_residuals(ptype, **kw)
    plt.semilogy(range(len(r)), r, "o-", ms=3, label=ptype)
plt.xlabel("CG iteration"); plt.ylabel("residual norm")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## 4. A tempting trap — the non-scalable preconditioner

`local` (Jacobi) is cheap and looks fine on a coarse mesh. But watch what
happens as we **refine**: its iteration count keeps climbing, while `multigrid`
stays essentially constant. *That* is what "optimal preconditioner" means — and
why the cheap choice is a trap for large problems.

In [ ]:
print(f"{'maxh':>6} {'ndof':>7} {'Jacobi':>8} {'multigrid':>10}")
for h in [0.08, 0.04, 0.02]:
    m = Lshape(h)
    _, it_j = solve_cg(m, "local")
    nd, it_mg = solve_cg(m, "multigrid", inverse="sparsecholesky")
    print(f"{h:>6} {nd:>7} {it_j:>8} {it_mg:>10}")

## 5. Static condensation

Interior (bubble) dofs can be eliminated element-by-element before solving
(`condense=True`), shrinking the global system. After solving the reduced
system we **reconstruct** the interior — forget those two lines and the
solution is silently wrong inside the elements!

In [ ]:
a = BilinearForm(grad(u)*grad(v)*dx, condense=True)
pre = Preconditioner(a, "multigrid", inverse="sparsecholesky")
a.Assemble()
inv = CGSolver(a.mat, pre.mat, maxiter=5000, tol=1e-10)
gfu.vec.data = inv * f.vec
gfu.vec.data += a.harmonic_extension * gfu.vec     # reconstruct interior (1/2)
gfu.vec.data += a.inner_solve * f.vec              # reconstruct interior (2/2)
print("condensed CG:", inv.iterations, "iterations")
Draw(gfu, mesh)

:::{dropdown} 🧠 Quiz — why does Jacobi blow up but multigrid doesn't?
The condition number of the stiffness matrix grows like $h^{-2}$ under
refinement, and CG with Jacobi needs $\mathcal{O}(\sqrt{\kappa})\sim
\mathcal{O}(h^{-1})$ iterations. Multigrid is *spectrally optimal*: it tackles
every error frequency on its own grid level, so the iteration count is bounded
**independently of the mesh size** — the gold standard for large problems.
:::

Next: **postprocessing** — extracting numbers, fluxes and pictures from a
solution.